# MonoDGP M59d: 2D-transformer layer diagnostic

> **LATEST NOTEBOOK REVISION — M59d-2026-09-18-r2**
>
> Use this revision for the current M59d run. The clone cell deliberately builds the Git URL from fragments and prints its `repr`; the expected value is `https://github.com/PuFanqi23/MonoDETR.git`. Do not paste a rendered Markdown link such as `[https://...](https://...)` into a Python string.

Run these cells on a Colab GPU after a session restart. This notebook restores the exact M59b MonoDGP checkout, reapplies all compatibility patches, rebuilds the frozen dataset view, and exports a new FP32 diagnostic package. It performs no training and does not authorize FP16, quantization, or device deployment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from collections import deque
from pathlib import Path
import json, os, shlex, shutil, subprocess, sys

# LATEST REVISION: M59d-2026-09-18-r2. Do not replace these URL fragments with a rendered Markdown link.
MOBILE_REPO = Path('/content/mobile_adas3d')
def github_url(owner, repository):
    return ''.join(['h', 't', 't', 'p', 's', ':', '/', '/', 'g', 'i', 't', 'h', 'u', 'b', '.', 'c', 'o', 'm', '/', owner, '/', repository, '.', 'g', 'i', 't'])
MOBILE_URL = github_url('Ali-RT', 'mobile_adas3d')
MONODETR_URL = github_url('PuFanqi23', 'MonoDETR')
if '[' in MONODETR_URL or '](' in MONODETR_URL or not MONODETR_URL.endswith('/PuFanqi23/MonoDETR.git'):
    raise ValueError(f'Malformed MonoDETR clone URL: {MONODETR_URL!r}; rerun this latest setup cell instead of copying a rendered link')
MONODGP_REPO = Path('/content/MonoDETR_M59b')
MONODGP_COMMIT = 'aa059a18214aebf644510e7f0793971b403f9d14'
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
DATASET_ROOT = Path('/content/monodgp_kitti_m56d')
M59B_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m59b_coreml_diagnostic')
M59B_GATE = M59B_ROOT / 'm59b_coreml_diagnostic_export_gate.json'
OUTPUT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m59d_2d_transformer')
M59D_LOG = OUTPUT_ROOT / 'colab_logs/m59d_export.log'

def run_checked(command, cwd=None):
    command = [str(item) for item in command]
    print('+', shlex.join(command), flush=True)
    result = subprocess.run(command, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')

def run_streamed(command, cwd, log_path):
    command = [str(item) for item in command]
    print('+', shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=100)
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=str(cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            tail.append(line.rstrip())
        code = process.wait()
    if code:
        raise RuntimeError(f'Exit {code}; full log: {log_path}\n' + '\n'.join(tail))

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not MOBILE_REPO.is_dir():
    run_checked(['git', 'clone', MOBILE_URL, MOBILE_REPO])
print('Paths initialized.')

In [ ]:
# LATEST REVISION M59d-2026-09-18-r2: restore the exact disposable MonoDETR checkout and reviewed patches.
if MONODGP_REPO.exists() and not (MONODGP_REPO / '.git').is_dir():
    shutil.rmtree(MONODGP_REPO)
if not MONODGP_REPO.is_dir():
    run_checked(['git', 'clone', MONODETR_URL, MONODGP_REPO])
run_checked(['git', 'fetch', '--all'], cwd=MONODGP_REPO)
run_checked(['git', 'reset', '--hard', MONODGP_COMMIT], cwd=MONODGP_REPO)
run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'coremltools==9.0', 'pyyaml', 'scipy', 'opencv-python-headless', 'numba', 'scikit-image', 'scikit-learn', 'tqdm', 'ninja', 'timm==1.0.20', 'pandas'])
for patch in ('patch_monodetr_colab_compat.py', 'patch_monodetr_m54_training.py', 'patch_monodetr_m57_deformable_attention.py', 'patch_monodetr_m58_coreml.py'):
    run_checked([sys.executable, str(MOBILE_REPO / 'scripts' / patch), '--monodgp-repo', MONODGP_REPO], cwd=MOBILE_REPO)
M59D_SCRIPT = MOBILE_REPO / 'scripts/export_monodgp_m59d_2d_transformer.py'
if not M59D_SCRIPT.is_file():
    raise FileNotFoundError(f'M59d exporter missing from {MOBILE_REPO}; sync the updated mobile_adas3d checkout before continuing')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=MONODGP_REPO, text=True).strip()
assert commit == MONODGP_COMMIT, (commit, MONODGP_COMMIT)
print('MonoDETR checkout restored:', commit)

In [ ]:
# Rebuild the exact M56d dataset view and validate the frozen M59b gate.
if not M59B_GATE.is_file():
    raise FileNotFoundError(M59B_GATE)
m59b_gate = json.loads(M59B_GATE.read_text())
if not (m59b_gate.get('complete') and m59b_gate.get('all_export_gates_passed') and m59b_gate.get('diagnostic_runtime_authorized')):
    raise RuntimeError(m59b_gate)

def first_dir(*paths):
    for path in paths:
        path = Path(path)
        if path.is_dir():
            return path
    return None

sources = {
    'image_2': first_dir(LOCAL_DATASET_ROOT / 'training/image_2', LOCAL_DATASET_ROOT / 'training/image_02', DRIVE_DATASET_ROOT / 'training/image_2', DRIVE_DATASET_ROOT / 'training/image_02'),
    'label_2': first_dir(LOCAL_DATASET_ROOT / 'training/label_2', LOCAL_DATASET_ROOT / 'training/label_02', DRIVE_DATASET_ROOT / 'training/label_2', DRIVE_DATASET_ROOT / 'training/label_02'),
    'calib': first_dir(LOCAL_DATASET_ROOT / 'training/calib', DRIVE_DATASET_ROOT / 'training/calib'),
}
if any(value is None for value in sources.values()):
    raise FileNotFoundError(f'KITTI source directories not found: {sources}')
(DATASET_ROOT / 'training').mkdir(parents=True, exist_ok=True)
(DATASET_ROOT / 'ImageSets').mkdir(parents=True, exist_ok=True)
for name, source in sources.items():
    link = DATASET_ROOT / 'training' / name
    if link.is_symlink() and link.resolve() == source.resolve():
        continue
    if link.exists() or link.is_symlink():
        raise RuntimeError(f'Unexpected existing dataset path: {link}')
    link.symlink_to(source, target_is_directory=True)
for split in ('train', 'val'):
    source = SPLIT_DIR / f'{split}.txt'
    if not source.is_file():
        raise FileNotFoundError(source)
    (DATASET_ROOT / 'ImageSets' / f'{split}.txt').write_text(source.read_text())
assert len((SPLIT_DIR / 'val.txt').read_text().splitlines()) == 3769
print('Dataset view ready:', DATASET_ROOT)

In [ ]:
# Export the layer-by-layer 2D-transformer diagnostic.
run_streamed([
    sys.executable, '-u', str(M59D_SCRIPT),
    '--monodgp-repo', MONODGP_REPO,
    '--m59b-export-gate', M59B_GATE,
    '--dataset-root', DATASET_ROOT,
    '--split-dir', SPLIT_DIR,
    '--output-dir', OUTPUT_ROOT,
], cwd=MOBILE_REPO, log_path=M59D_LOG)
report = json.loads((OUTPUT_ROOT / 'm59d_2d_transformer_export_gate.json').read_text())
if not report.get('complete') or not report.get('all_export_gates_passed'):
    raise RuntimeError(report)
print(json.dumps(report, indent=2))
print('STOP: copy the M59d package, reference I/O, and gate JSON to macOS for validation.')

## Stop point

Copy `MonoDGP_M59d_2d_transformer_fp32.mlpackage`, `m59d_2d_transformer_reference_io.npz`, and `m59d_2d_transformer_export_gate.json` to macOS. Run `scripts/validate_monodgp_m59d_macos.py` there. Do not run FP16, quantization, or physical-device testing until the first failing encoder/decoder layer is understood.